# Mean–Variance Portfolio Optimization
### Improvements: Ledoit-Wolf shrinkage · Black-Litterman returns · Sharpe tangency · Even frontier sweep · Out-of-sample validation

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy.optimize as opt
from sklearn.covariance import LedoitWolf

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1 · Data Download & Train / Test Split

In [ ]:
TICKERS   = ["SPY", "QQQ", "IWM", "EFA", "EEM", "TLT"]
RF        = 0.052          # annualised risk-free rate (approx 3-mo T-bill 2024)
W_MAX     = 0.30           # max weight per asset
TRAIN_END = "2023-01-01"   # train on 2018-2022, test on 2023-2025

raw = yf.download(TICKERS, start="2018-01-01", end="2025-01-01",
                  auto_adjust=True)["Close"]
prices = raw.dropna()
returns_all = prices.pct_change().dropna()

# ── Train / test split ─────────────────────────────────────────────────────
ret_train = returns_all.loc[:TRAIN_END]
ret_test  = returns_all.loc[TRAIN_END:]

print(f"Train: {ret_train.index[0].date()} → {ret_train.index[-1].date()}  ({len(ret_train)} days)")
print(f"Test : {ret_test.index[0].date()}  → {ret_test.index[-1].date()}   ({len(ret_test)} days)")

## 2 · Estimation: Ledoit-Wolf Shrinkage + Black-Litterman Expected Returns

**Why shrink the covariance?**  
The sample covariance matrix with $T \approx 1250$ days and $N=6$ assets is already well-conditioned, but shrinkage still reduces in-sample over-fitting — especially for the off-diagonal correlations that drive diversification.

**Why Black-Litterman?**  
Raw sample means $\hat{\mu}$ have enormous estimation error (Sharpe ratio of the sample mean ≈ 0.3 after 5 years). BL anchors returns to market-cap-implied **equilibrium** returns $\Pi = \delta \Sigma w^{mkt}$, then blends in your views. With no active views, BL simply uses $\Pi$ — which is already far more stable than $\hat{\mu}$.

In [ ]:
# ── Ledoit-Wolf shrinkage covariance ───────────────────────────────────────
lw = LedoitWolf().fit(ret_train)
Sigma = pd.DataFrame(
    lw.covariance_ * 252,          # annualise
    index=TICKERS, columns=TICKERS
)
print(f"Ledoit-Wolf shrinkage coefficient: {lw.shrinkage_:.3f}")

# ── Approximate market-cap weights (manual, as of early 2023) ──────────────
# SPY ≈ large cap blend, QQQ ≈ tech heavy, IWM ≈ small cap,
# EFA ≈ intl developed, EEM ≈ emerging, TLT ≈ long bonds
mkt_caps = np.array([0.40, 0.20, 0.10, 0.15, 0.08, 0.07])
w_mkt = mkt_caps / mkt_caps.sum()

# ── Black-Litterman equilibrium returns ────────────────────────────────────
# δ = market risk-aversion, implied from Sharpe of market portfolio
sigma_mkt = np.sqrt(w_mkt @ Sigma.values @ w_mkt)
ret_mkt   = ret_train.values @ w_mkt          # daily market return
mu_mkt    = ret_mkt.mean() * 252
delta     = (mu_mkt - RF) / sigma_mkt**2      # implied risk aversion

Pi = delta * Sigma.values @ w_mkt             # equilibrium (prior) returns
mu = pd.Series(Pi, index=TICKERS)

print(f"\nImplied δ (risk aversion): {delta:.2f}")
print("\nBL equilibrium expected returns (annualised):")
print(mu.sort_values(ascending=False).map("{:.2%}".format))

## 3 · Efficient Frontier via Target-Return Sweep

Instead of sweeping a scalar $\lambda$, we pin a **target return** $\mu^*$ and minimise variance subject to $\mu^\top w = \mu^*$. This gives evenly spaced points along the frontier.

In [ ]:
def min_variance_for_target(mu_arr, Sigma_arr, target_ret, w_max=W_MAX):
    """Minimise portfolio variance subject to a target expected return."""
    n = len(mu_arr)
    constraints = [
        {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
        {'type': 'eq', 'fun': lambda w: mu_arr @ w - target_ret},
    ]
    bounds = [(0.0, w_max)] * n
    w0 = np.ones(n) / n
    res = opt.minimize(
        lambda w: w @ Sigma_arr @ w,
        w0, method='SLSQP', bounds=bounds, constraints=constraints
    )
    if not res.success:
        return None, None, None
    vol = np.sqrt(res.x @ Sigma_arr @ res.x)
    return res.x, vol, mu_arr @ res.x


def min_variance_portfolio(mu_arr, Sigma_arr, w_max=W_MAX):
    """Global minimum-variance portfolio (no return target)."""
    n = len(mu_arr)
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = [(0.0, w_max)] * n
    res = opt.minimize(
        lambda w: w @ Sigma_arr @ w,
        np.ones(n)/n, method='SLSQP', bounds=bounds, constraints=constraints
    )
    vol = np.sqrt(res.x @ Sigma_arr @ res.x)
    return res.x, vol, mu_arr @ res.x


def max_sharpe_portfolio(mu_arr, Sigma_arr, rf=RF, w_max=W_MAX):
    """Tangency portfolio — maximise Sharpe ratio."""
    n = len(mu_arr)
    def neg_sharpe(w):
        r   = mu_arr @ w
        vol = np.sqrt(w @ Sigma_arr @ w)
        return -(r - rf) / vol
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    bounds = [(0.0, w_max)] * n
    res = opt.minimize(
        neg_sharpe, np.ones(n)/n,
        method='SLSQP', bounds=bounds, constraints=constraints
    )
    vol = np.sqrt(res.x @ Sigma_arr @ res.x)
    ret = mu_arr @ res.x
    return res.x, vol, ret


# ── Build frontier ─────────────────────────────────────────────────────────
mu_arr    = mu.values
Sigma_arr = Sigma.values

w_mv, vol_mv, ret_mv = min_variance_portfolio(mu_arr, Sigma_arr)
w_tan, vol_tan, ret_tan = max_sharpe_portfolio(mu_arr, Sigma_arr)

# Sweep from min-variance return up to max individual asset return
ret_low  = ret_mv
ret_high = mu_arr.max() * 0.95   # stay inside feasible region
targets  = np.linspace(ret_low, ret_high, 60)

frontier_vols, frontier_rets, frontier_ws = [], [], []
for t in targets:
    w, v, r = min_variance_for_target(mu_arr, Sigma_arr, t)
    if w is not None:
        frontier_vols.append(v)
        frontier_rets.append(r)
        frontier_ws.append(w)

frontier_vols = np.array(frontier_vols)
frontier_rets = np.array(frontier_rets)
sharpes       = (frontier_rets - RF) / frontier_vols

print(f"Min-variance portfolio  → vol={vol_mv:.2%}  ret={ret_mv:.2%}")
print(f"Max-Sharpe (tangency)   → vol={vol_tan:.2%}  ret={ret_tan:.2%}  SR={(ret_tan-RF)/vol_tan:.2f}")

## 4 · Visualization — Frontier + Sharpe Heatmap

In [ ]:
# ── Per-asset stats (train) ────────────────────────────────────────────────
asset_vols = np.sqrt(np.diag(Sigma_arr))
asset_rets = mu_arr

fig, ax = plt.subplots(figsize=(10, 6))

# Frontier coloured by Sharpe ratio
sc = ax.scatter(
    frontier_vols, frontier_rets,
    c=sharpes, cmap='RdYlGn', s=40, zorder=3, label='_nolegend_'
)
cbar = fig.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label('Sharpe Ratio', fontsize=10)

# Individual assets
for i, t in enumerate(TICKERS):
    ax.scatter(asset_vols[i], asset_rets[i], marker='D',
               s=70, zorder=4, color='steelblue')
    ax.annotate(t, (asset_vols[i], asset_rets[i]),
                textcoords='offset points', xytext=(6, 3), fontsize=9)

# Special portfolios
ax.scatter(vol_mv, ret_mv, marker='*', s=220, color='royalblue',
           zorder=5, label=f'Min Variance  SR={(ret_mv-RF)/vol_mv:.2f}')
ax.scatter(vol_tan, ret_tan, marker='*', s=220, color='gold',
           edgecolors='darkorange', linewidths=0.8,
           zorder=5, label=f'Max Sharpe (Tangency)  SR={(ret_tan-RF)/vol_tan:.2f}')

# Capital Market Line
x_cml = np.array([0, frontier_vols.max() * 1.05])
slope  = (ret_tan - RF) / vol_tan
ax.plot(x_cml, RF + slope * x_cml, '--', color='darkorange',
        lw=1.2, label='Capital Market Line')

ax.axhline(RF, color='gray', lw=0.8, linestyle=':', label=f'Risk-free rate {RF:.1%}')
ax.set_xlabel('Portfolio Volatility (σ)', fontsize=11)
ax.set_ylabel('Expected Return (μ)', fontsize=11)
ax.set_title('Constrained Efficient Frontier — BL Returns + Ledoit-Wolf Σ', fontsize=13)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5 · Tangency Portfolio Weights

In [ ]:
tan_series = pd.Series(w_tan, index=TICKERS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ['#2196F3' if w > 0.01 else '#B0BEC5' for w in tan_series]
bars = ax.barh(tan_series.index, tan_series.values, color=colors)
for bar, val in zip(bars, tan_series.values):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=9)
ax.axvline(0, color='black', lw=0.5)
ax.set_xlim(0, W_MAX + 0.08)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_title('Max-Sharpe (Tangency) Portfolio Weights', fontsize=12)
plt.tight_layout()
plt.show()

print("\nTangency weights:")
print(tan_series.map("{:.2%}".format))
print(f"\nExpected return : {ret_tan:.2%}")
print(f"Expected vol    : {vol_tan:.2%}")
print(f"Sharpe ratio    : {(ret_tan - RF) / vol_tan:.2f}")

## 6 · Out-of-Sample Validation (2023–2025)

We compare three portfolios on **held-out** data:

| Portfolio | Construction |
|---|---|
| **Tangency** | Max-Sharpe from training period |
| **Min-Variance** | Lowest-vol portfolio from training |
| **Equal-Weight** | 1/N benchmark |

In [ ]:
def portfolio_stats(weights, returns_df, rf_daily=RF/252):
    """Annualised return, vol, Sharpe, and max drawdown from daily returns."""
    port_ret = returns_df.values @ weights
    ann_ret  = port_ret.mean() * 252
    ann_vol  = port_ret.std()  * np.sqrt(252)
    sharpe   = (ann_ret - RF) / ann_vol
    # Max drawdown
    cum  = (1 + port_ret).cumprod()
    roll_max = np.maximum.accumulate(cum)
    dd   = (cum - roll_max) / roll_max
    mdd  = dd.min()
    return ann_ret, ann_vol, sharpe, mdd, port_ret


w_ew = np.ones(len(TICKERS)) / len(TICKERS)

portfolios = {
    'Tangency (Max-Sharpe)': w_tan,
    'Min-Variance':          w_mv,
    'Equal-Weight (1/N)':    w_ew,
}

results = {}
for name, w in portfolios.items():
    r, v, sr, mdd, daily = portfolio_stats(w, ret_test)
    results[name] = {'Return': r, 'Volatility': v, 'Sharpe': sr, 'Max DD': mdd, 'daily': daily}

# ── Summary table ──────────────────────────────────────────────────────────
summary = pd.DataFrame(results).T.drop(columns='daily')
fmt = {'Return': '{:.2%}'.format, 'Volatility': '{:.2%}'.format,
       'Sharpe': '{:.2f}'.format, 'Max DD': '{:.2%}'.format}
print("Out-of-sample performance (2023–2025):")
print(summary.style.format(fmt).to_string())

## 7 · Cumulative Return — Out-of-Sample

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors_map = {
    'Tangency (Max-Sharpe)': 'gold',
    'Min-Variance':          'royalblue',
    'Equal-Weight (1/N)':    'dimgray',
}

for name, w in portfolios.items():
    daily = results[name]['daily']
    cum   = (1 + daily).cumprod()
    ax.plot(ret_test.index, cum, label=name, lw=1.8, color=colors_map[name])

ax.axhline(1.0, color='black', lw=0.5, linestyle=':')
ax.set_title('Out-of-Sample Cumulative Return (2023–2025)', fontsize=13)
ax.set_ylabel('Growth of $1', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'${y:.2f}'))
ax.legend(fontsize=10)
ax.grid(axis='y', lw=0.4, alpha=0.6)
plt.tight_layout()
plt.show()